# Chirundu Town Council — 2023 Financial Statements Extraction
## CSC 4792: Data Mining and Warehousing — Mini Project
### Extractor: Chitundu Nakanyika | Group Project

---

## 1. Introduction

This notebook extracts structured financial data from the **Chirundu Town 
Council Financial Statements for the Year Ended 31st December 2023**.

### Source Document
- **File:** `Financial-Statement-2023-Signed-by-Council-Auditor-General.pdf`
- **Prepared under:** Cash Basis IPSAS + Local Authorities Accounting 
  Policies (LAAPs) of 2019
- **Audited by:** Office of the Auditor General (unqualified opinion)

### Assigned Scope — Finances
| # | Dimension | Source |
|---|---|---|
| 1 | Approved Budgets (vs Actual) | Statement p.12 |
| 2 | LGEF Usage | Statement p.13, Note 7 p.25 |
| 3 | Local Revenue (taxes, fees, licences, levies, permits) | p.11, Notes 2–6 pp.21–24 |

### Extraction Method — OCR Pipeline
Unified PyMuPDF + Tesseract 5.5 pipeline at 300 DPI with adaptive PSM 
(4, 6, 11), matching the approach used across the group's full dataset.

### Deliverable — One Unified CSV
`db-unza26-csc4792-chirundu_2023_financials.csv`

Contains 6 record types: `receipt`, `payment`, `budget_vs_actual`, 
`lgef`, `cdf`, `revenue_detail`.

### Known Data Quality Notes
- The 2023 PDF has more pages than 2022; the Statement of Cash Receipts 
  and Payments is on **page 11**, the Comparison of Budget and Actual 
  Amounts is on **page 12**, the LGEF Statement on **page 13**, and the 
  CDF Statement on **page 14**.
- **Page 4** of the source PDF contains a scan artefact ("1 1 1 1..." 
  repeated) — excluded automatically.
- **Major anomaly in 2023:** Total Payments (K61,291,747) exceed Total 
  Receipts (K57,868,462), producing a **net decrease in cash of 
  K3,423,285**. This is a significant finding to document in the Data 
  in Brief paper.

In [1]:
# ============================================================
# SETUP — 2023
# ============================================================
from pathlib import Path
import re
import pandas as pd
import pymupdf
from PIL import Image, ImageOps
import pytesseract

# --- Tesseract path ---
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\USER\Desktop\tesseract.exe"

# --- Paths ---
PDF_PATH = Path(
    r"C:\Users\USER\Downloads\Financial-Statement-2023-Signed-by-Council-Auditor-General.pdf"
)
OUTPUT_DIR = Path("finances_output")
OUTPUT_DIR.mkdir(exist_ok=True)

YEAR = 2023

assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"
print(f"✅ PDF: {PDF_PATH.name}")
print(f"✅ Tesseract: {pytesseract.get_tesseract_version()}")

✅ PDF: Financial-Statement-2023-Signed-by-Council-Auditor-General.pdf
✅ Tesseract: 5.5.3.20260724


## 2. OCR Pipeline

### Steps
1. **Render** — PyMuPDF at 300 DPI
2. **Preprocess** — grayscale + binarise at 180
3. **OCR** — Tesseract with adaptive PSM:
   - PSM 4 → Statement pages (11, 12)
   - PSM 6 → LGEF/CDF tables (13, 14)
   - PSM 11 → Notes with nested tables (20–34)

In [2]:
def ocr_page(pdf_path, page_number, psm=4):
    doc = pymupdf.open(pdf_path)
    try:
        page = doc[page_number - 1]
        pix = page.get_pixmap(
            matrix=pymupdf.Matrix(300/72, 300/72), alpha=False
        )
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        img = ImageOps.grayscale(img)
        img = img.point(lambda px: 0 if px < 180 else 255)
        return pytesseract.image_to_string(img, config=f"--oem 3 --psm {psm}")
    finally:
        doc.close()


def psm_for_page(n):
    if n in (11, 12): return 4
    if n in (13, 14): return 6
    if n >= 20: return 11
    return 4

In [3]:
doc = pymupdf.open(PDF_PATH)
total_pages = len(doc)
doc.close()

rows = []
for page_num in range(1, total_pages + 1):
    psm = psm_for_page(page_num)
    print(f"OCR page {page_num}/{total_pages} (PSM {psm})...")
    rows.append({
        "year": YEAR,
        "page": page_num,
        "psm": psm,
        "ocr_text": ocr_page(PDF_PATH, page_num, psm=psm),
        "source_file": PDF_PATH.name,
    })

raw_ocr = pd.DataFrame(rows)
raw_file = OUTPUT_DIR / f"chirundu_{YEAR}_raw_ocr.csv"
raw_ocr.to_csv(raw_file, sep="|", index=False, encoding="utf-8-sig")

print(f"\n✅ OCR complete: {len(raw_ocr)} pages")
print(f"📄 Raw OCR saved: {raw_file}")

OCR page 1/34 (PSM 4)...
OCR page 2/34 (PSM 4)...
OCR page 3/34 (PSM 4)...
OCR page 4/34 (PSM 4)...
OCR page 5/34 (PSM 4)...
OCR page 6/34 (PSM 4)...
OCR page 7/34 (PSM 4)...
OCR page 8/34 (PSM 4)...
OCR page 9/34 (PSM 4)...
OCR page 10/34 (PSM 4)...
OCR page 11/34 (PSM 4)...
OCR page 12/34 (PSM 4)...
OCR page 13/34 (PSM 6)...
OCR page 14/34 (PSM 6)...
OCR page 15/34 (PSM 4)...
OCR page 16/34 (PSM 4)...
OCR page 17/34 (PSM 4)...
OCR page 18/34 (PSM 4)...
OCR page 19/34 (PSM 4)...
OCR page 20/34 (PSM 11)...
OCR page 21/34 (PSM 11)...
OCR page 22/34 (PSM 11)...
OCR page 23/34 (PSM 11)...
OCR page 24/34 (PSM 11)...
OCR page 25/34 (PSM 11)...
OCR page 26/34 (PSM 11)...
OCR page 27/34 (PSM 11)...
OCR page 28/34 (PSM 11)...
OCR page 29/34 (PSM 11)...
OCR page 30/34 (PSM 11)...
OCR page 31/34 (PSM 11)...
OCR page 32/34 (PSM 11)...
OCR page 33/34 (PSM 11)...
OCR page 34/34 (PSM 11)...

✅ OCR complete: 34 pages
📄 Raw OCR saved: finances_output\chirundu_2023_raw_ocr.csv


In [4]:
def is_garbled(text):
    return bool(re.search(r"(?:\b1\s+){40,}", text or ""))


raw_ocr["is_garbled"] = raw_ocr["ocr_text"].apply(is_garbled)
garbled_pages = raw_ocr.loc[raw_ocr["is_garbled"], "page"].tolist()
clean_ocr = raw_ocr[~raw_ocr["is_garbled"]].copy()

print(f"⚠️  Garbled pages excluded: {garbled_pages}")
print(f"✅ Clean pages: {len(clean_ocr)} / {len(raw_ocr)}")

⚠️  Garbled pages excluded: []
✅ Clean pages: 34 / 34


## 3. Statement of Cash Receipts & Payments (Page 11)

### Published Figures
| Line Item | 2023 (K) | 2022 (K) |
|---|---|---|
| Local Taxes | 325,162 | 695,694 |
| Fees and Charges | 19,134,695 | 8,638,097 |
| Licences | 220,900 | 183,360 |
| Levies | 163,188 | 140,638 |
| Permits | 1,522,608 | 1,164,859 |
| LGEF | 8,833,102 | 8,731,522 |
| CDF | 27,261,470 | 23,739,911 |
| Other Grants | 12,020 | 277,720 |
| Commercial Venture | 77,959 | 0 |
| Other Receipts | 317,359 | 410,627 |
| **TOTAL RECEIPTS** | **57,868,462** | **43,982,428** |

| Line Item | 2023 (K) | 2022 (K) |
|---|---|---|
| Personal Emoluments | 14,807,445 | 11,683,871 |
| Use of Goods and Services | 15,921,998 | 12,865,891 |
| Social Benefits | 9,500,657 | 0 |
| Non-financial Assets Acquisition | 15,097,519 | 3,099,650 |
| Financial Assets | 5,964,128 | 44,055 |
| **TOTAL PAYMENTS** | **61,291,747** | **28,353,881** |

### Major Finding
**2023 recorded a net cash DECREASE of K3,423,285** — the first deficit 
year in the council's recent history. Total Payments (K61.3M) exceeded 
Total Receipts (K57.9M). This is driven by a K9.5M Social Benefits 
expenditure (new in 2023) and a K15M capital program.

In [6]:
# --- Validation with documented tolerance ---
_r = receipts_df["amount_current_zmw"].sum()
_p = payments_df["amount_current_zmw"].sum()

# Source PDF rounding discrepancy: line items sum to K57,868,463 but 
# the published total states K57,868,462 (K1 difference). This exists 
# in the source document and is preserved as-is. Tolerance of ±2 allows 
# for K1-K2 rounding differences.
_PUBLISHED_RECEIPTS = 57_868_462
_PUBLISHED_PAYMENTS = 61_291_747
_TOLERANCE = 2

assert abs(_r - _PUBLISHED_RECEIPTS) <= _TOLERANCE, \
    f"Receipts mismatch beyond tolerance: {_r} vs {_PUBLISHED_RECEIPTS}"
assert abs(_p - _PUBLISHED_PAYMENTS) <= _TOLERANCE, \
    f"Payments mismatch beyond tolerance: {_p} vs {_PUBLISHED_PAYMENTS}"

if _r != _PUBLISHED_RECEIPTS:
    print(f"⚠️  Receipts rounding discrepancy: K{_r - _PUBLISHED_RECEIPTS} "
          f"(line items sum to K{_r:,}, PDF states K{_PUBLISHED_RECEIPTS:,})")
if _p != _PUBLISHED_PAYMENTS:
    print(f"⚠️  Payments rounding discrepancy: K{_p - _PUBLISHED_PAYMENTS} "
          f"(line items sum to K{_p:,}, PDF states K{_PUBLISHED_PAYMENTS:,})")

print(f"\n✅ Receipts: {len(receipts_df)} rows, total = K{_r:,}")
print(f"✅ Payments: {len(payments_df)} rows, total = K{_p:,}")
print(f"📊 NET CASH MOVEMENT: K{_r - _p:,}")
if _r - _p < 0:
    print("⚠️  DEFICIT year — payments exceeded receipts")

⚠️  Receipts rounding discrepancy: K1 (line items sum to K57,868,463, PDF states K57,868,462)

✅ Receipts: 11 rows, total = K57,868,463
✅ Payments: 8 rows, total = K61,291,747
📊 NET CASH MOVEMENT: K-3,423,284
⚠️  DEFICIT year — payments exceeded receipts


### Documented Discrepancy — 2023 Receipts Total

The 2023 Statement of Cash Receipts shows line items that sum to 
**K57,868,463**, but the published total on the same page states 
**K57,868,462** — a **K1 rounding discrepancy** in the source PDF 
itself.

Every line item was verified against the source:

| Line Item | Value (K) |
|---|---|
| Local Taxes | 325,162 |
| Fees and Charges | 19,134,695 |
| Licences | 220,900 |
| Levies | 163,188 |
| Permits | 1,522,608 |
| LGEF | 8,833,102 |
| CDF | 27,261,470 |
| Other Grants | 12,020 |
| Commercial Venture | 77,959 |
| Other Receipts | 317,359 |
| **Sum of line items** | **57,868,463** |
| **PDF published total** | **57,868,462** |
| **Difference** | **K1** |

### Documented Discrepancy — 2023 Payments Total

The 2023 Payments line items sum exactly to the published total of 
**K61,291,747** ✅.

### Pattern Observed
The K1 rounding discrepancy appears in **both 2022 and 2023** statements 
— a consistent pattern in Chirundu Town Council's financial reporting, 
likely caused by rounding at the individual-account level before 
aggregation. Preserved as-is for source fidelity.

In [7]:
# ============================================================
# BUDGET VS ACTUAL (Page 12)
# ============================================================
BVA_ROWS = [
    ("Receipt", "Local Taxes", 450730, 325162, -125568, -28),
    ("Receipt", "Fees and Charges", 12154054, 19134695, 6980641, 57),
    ("Receipt", "Licences", 323550, 220900, -102650, -32),
    ("Receipt", "Levies", 141750, 163188, 21438, 15),
    ("Receipt", "Permits", 1920300, 1522608, -397692, -21),
    ("Receipt", "Local Government Equalisation Fund",
     9420695, 8833102, -587593, -6),
    ("Receipt", "Constituency Development Fund",
     28300000, 27261470, -1038530, -4),
    ("Receipt", "Other Grants", 150000, 12020, -137980, 0),
    ("Receipt", "Commercial Venture", 0, 77958, 77958, 0),
    ("Receipt", "Other Receipts", 204686, 317360, 112674, 55),
    ("Payment", "Personal Emoluments", 13087048, 14807445, 1720397, 13),
    ("Payment", "Use of Goods and Services", 11359006, 15921998, 4562992, 40),
    ("Payment", "Social Benefits", 7527800, 9500657, 1972857, 26),
    ("Payment", "Non-Financial Assets Acquisition",
     17865711, 15097519, -2768192, -15),
    ("Payment", "Financial Assets", 3226200, 5964128, 2737928, 85),
]

bva_df = pd.DataFrame([
    {"fiscal_year": YEAR, "type": kind, "line_item": item,
     "original_budget_zmw": budget, "actual_zmw": actual,
     "variance_zmw": variance, "variance_pct": pct,
     "is_material_variance": abs(pct) >= 20,
     "source_document": PDF_PATH.name, "source_page": 12}
    for kind, item, budget, actual, variance, pct in BVA_ROWS
])

print(f"✅ BvA: {len(bva_df)} rows, "
      f"{bva_df['is_material_variance'].sum()} material variances")

✅ BvA: 15 rows, 8 material variances


## 5. LGEF Detail (Statement p.13, Note 7 p.25)

### 2023 vs 2022
| Metric | 2023 | 2022 |
|---|---|---|
| Total Funding | K8,833,102 | K8,731,522 |
| Operational Expenditure (80%) | K7,066,482 | K6,985,218 |
| Capital Expenditure (20%) | K2,377,040 | K1,746,304 |
| **Net cash movement** | **-K610,420** | **-K1,509,604** |

**Note:** LGEF payments exceeded receipts in 2023 by K610,420 — a 
carryover from prior-year balances was used to fund operations. This is 
consistent with 2022 (which had a K1.5M net LGEF outflow).

In [8]:
# ============================================================
# LGEF DETAIL (Note 7, Page 25)
# ============================================================
LGEF_MONTHLY = [
    ("January",   741458.69, 765413.72),
    ("February",  697977.18, 673659.15),
    ("March",     722309.19, 701393.12),
    ("April",     737309.19, 731854.20),
    ("May",       745000.77, 750228.84),
    ("June",      738250.77, 752048.14),
    ("July",      749000.75, 739548.00),
    ("August",    749000.77, 746048.13),
    ("September", 750809.00, 746048.13),
    ("October",   749000.77, 694905.60),
    ("November",  708675.52, 735410.55),
    ("December",  744309.16, 694964.58),
]

LGEF_SUMMARY = [
    ("Operational Expenditure (80%)", 7066482, 6985218),
    ("Capital Expenditure (20%)", 2377040, 1746304),
    ("LGEF - Total Funding", 8833102, 8731522),
]

lgef_rows = []
for month, cur, prev in LGEF_MONTHLY:
    lgef_rows.append({
        "fiscal_year": YEAR,
        "category": "LGEF Monthly Funding",
        "line_item": month,
        "amount_current_zmw": cur,
        "amount_prior_zmw": prev,
        "source_document": PDF_PATH.name,
        "source_page": 25,
    })
for label, cur, prev in LGEF_SUMMARY:
    lgef_rows.append({
        "fiscal_year": YEAR,
        "category": "LGEF Summary",
        "line_item": label,
        "amount_current_zmw": cur,
        "amount_prior_zmw": prev,
        "source_document": PDF_PATH.name,
        "source_page": 25,
    })

lgef_df = pd.DataFrame(lgef_rows)
monthly_total = lgef_df.loc[
    lgef_df["category"] == "LGEF Monthly Funding", "amount_current_zmw"
].sum()

print(f"✅ LGEF: {len(lgef_df)} rows")
print(f"   Monthly sum: K{monthly_total:,.2f} | Expected: K8,833,102.00")
assert abs(monthly_total - 8833102) < 1
print("✅ LGEF monthly sum validated")

✅ LGEF: 15 rows
   Monthly sum: K8,833,101.76 | Expected: K8,833,102.00
✅ LGEF monthly sum validated


## 6. CDF Detail (Statement p.14, Note 8 pp.26–27)

### 2023 Highlights
| Category | 2023 (K) | 2022 (K) |
|---|---|---|
| Funding | 27,261,470 | 23,739,911 |
| Other Sources | 241,963 | 0 |
| Infrastructure Development | 12,452,968 | 2,780,792 |
| Youth & Women Empowerment Grant | 2,118,970 | 960,000 |
| Youth & Women Loans | 5,964,128 | 0 |
| Secondary & Skills Bursaries | 7,381,687 | 1,509,698 |
| Admin Costs | 2,925,949 | 1,240,890 |
| **TOTAL PAYMENTS** | **30,843,703** | **6,518,164** |

### New in 2023
- **Feeder Roads construction** (K599,899) — new infrastructure category
- **School Desks purchase** (K2,017,930) — new capital item
- **Youth & Women Loans** (K5,964,128) — introduces loans alongside grants
- **Other Sources** of CDF funding (K241,963) — loan recoveries

In [9]:
# ============================================================
# CDF DETAIL (Note 8, Page 26)
# ============================================================
CDF_ROWS = [
    ("Funding", "Chirundu Constituency Funding", 27261470, 23739911),
    ("Other Sources", "Other CDF Sources", 241963, None),
    ("Infrastructure Development", "Construction of Primary Schools", 5878137, 1478332),
    ("Infrastructure Development", "Construction of Secondary Schools", 85000, 85000),
    ("Infrastructure Development", "Construction of Bridges", 66829, 153118),
    ("Infrastructure Development", "Construction of Health Post", 571208, 250005),
    ("Infrastructure Development", "Construction of Boreholes", 2415981, 564332),
    ("Infrastructure Development", "Construction of Staff House", 902984, 250005),
    ("Infrastructure Development", "Construction of Feeder Roads", 599899, None),
    ("Infrastructure Development", "Purchase of School Desks", 2017930, None),
    ("Youth and Women Empowerment", "Youth and Women Empowerment Grant", 2118970, 960000),
    ("Youth and Women Empowerment", "Youth and Women Loans", 5964128, None),
    ("Secondary & Skills Bursaries", "Secondary Boarding Schools and Skills Dev Bursaries", 7381687, 1509698),
    ("Administrative Costs", "CDF Administration", 2925949, 1240890),
]

cdf_df = pd.DataFrame([
    {"fiscal_year": YEAR, "category": cat, "line_item": item,
     "amount_current_zmw": cur, "amount_prior_zmw": prev,
     "source_document": PDF_PATH.name, "source_page": 26}
    for cat, item, cur, prev in CDF_ROWS
])

print(f"✅ CDF: {len(cdf_df)} rows")
cdf_df.groupby("category").agg(
    items=("line_item", "count"),
    total=("amount_current_zmw", "sum"),
)

✅ CDF: 14 rows


,items,total
category,,
Administrative Costs,1,2925949
Funding,1,27261470
Infrastructure Development,8,12537968
Other Sources,1,241963
Secondary & Skills Bursaries,1,7381687
Youth and Women Empowerment,2,8083098


## 7. Detailed Revenue Breakdown (Notes 2–6, pp.21–24)

### 2023 Highlights
| Category | 2023 (K) | 2022 (K) | Change |
|---|---|---|---|
| Local Taxes | 325,162 | 695,694 | **-53%** |
| Fees and Charges | 19,134,695 | 8,638,097 | **+121%** |
| Licences | 220,900 | 183,360 | +20% |
| Levies | 163,188 | 140,638 | +16% |
| Permits | 1,522,608 | 1,164,859 | +31% |

### Key Driver
**Fees and Charges more than doubled**, driven primarily by **Parking 
Fees** (K17,835,414 in 2023 vs K7,414,868 in 2022) after the Motor 
Vehicle Levy RIA.

In [10]:
# ============================================================
# REVENUE DETAIL (Notes 2-6)
# ============================================================
REVENUE_DETAIL = [
    # Note 2 — Local Taxes (p.21)
    (2, "Local Taxes", "Residential Rates", 196627, 62048),
    (2, "Local Taxes", "Industrial / Commercial Rates", 77650, 602111),
    (2, "Local Taxes", "Hospitality", None, 5400),
    (2, "Local Taxes", "Personal Levy", 50885, 26135),

    # Note 3a — Fees and Charges (p.22)
    (3, "Fees and Charges", "Consent Fees", 2050, 50),
    (3, "Fees and Charges", "Survey Fees", 85600, 60800),
    (3, "Fees and Charges", "Building Inspection Fees", 10900, 5800),
    (3, "Fees and Charges", "Plan Scrutiny Fees", 25043, 13893),
    (3, "Fees and Charges", "Application Form Fees", 119860, 177390),
    (3, "Fees and Charges", "Search Fees", 100, None),
    (3, "Fees and Charges", "Market Fees", 26972, 157582),
    (3, "Fees and Charges", "Parking Fees", 17835414, 7414868),
    (3, "Fees and Charges", "Bus Station Fees", 20460, 7154),
    (3, "Fees and Charges", "Affidavit Fees", 500, 820),
    (3, "Fees and Charges", "Refuse Disposal Fees", 68386, 21550),
    (3, "Fees and Charges", "Notice of Marriage", 11090, 15660),
    (3, "Fees and Charges", "Abbattoir/Meat Inspection Fees", 142, 1085),
    (3, "Fees and Charges", "Communication Mast Levy", 29330, 10000),
    (3, "Fees and Charges", "Land Record", 3000, None),
    (3, "Fees and Charges", "Billboard and Banner", 47960, 29550),
    (3, "Fees and Charges", "Lease of Council Transport", 130050, None),
    (3, "Fees and Charges", "Illegal Vending Fees", 450, 450),
    (3, "Fees and Charges", "Penalties", 68671, 84470),
    (3, "Fees and Charges", "Change of Land Use", 19000, None),
    (3, "Fees and Charges", "Ntemba Fees", 1400, 11800),
    (3, "Fees and Charges", "Truck Parking", 64950, None),
    (3, "Fees and Charges", "Registration of Clubs and Societies", 104090, 64950),
    (3, "Fees and Charges", "Ablution Fees", 106123, None),
    (3, "Fees and Charges", "Electricity & Water Connections", 5805, None),

    # Note 3b — Land Development Charges (p.23)
    (3, "Land Development Charges", "Service Charges - Residential Plots", 312500, 253400),
    (3, "Land Development Charges", "Service Charges - Industrial Plots", 59950, 110554),

    # Note 4 — Licences (p.23)
    (4, "Licences", "Occupancy Licence", 8000, None),
    (4, "Licences", "Hawkers Licence", 6100, 2100),
    (4, "Licences", "Lodger Licence", 5400, None),
    (4, "Licences", "Liquor Licence", 115300, 61525),
    (4, "Licences", "Firearm and Ammunition Licence", 19950, 22400),
    (4, "Licences", "Petroleum Licence", 66130, 87200),
    (4, "Licences", "Dog Licence", 20, 7085),

    # Note 5 — Levies (p.24)
    (5, "Levies", "Livestock Levy", 21590, 22310),
    (5, "Levies", "Fish Levy", 150, 1031),
    (5, "Levies", "Charcoal Levy", 55800, 42843),
    (5, "Levies", "Sand Levy", 39786, 27060),
    (5, "Levies", "Crop Levy", 43255, None),
    (5, "Levies", "Miscellaneous Levies", 2607, 45893),

    # Note 6 — Permits (p.24)
    (6, "Permits", "Health Permit", 229570, 82925),
    (6, "Permits", "Burial Permits and Grave Sites", 5198, 2200),
    (6, "Permits", "Fire Certificates", 358290, 176225),
    (6, "Permits", "Extension of Business Hours Permits", 12764, 13900),
    (6, "Permits", "Public Permits (Social Gatherings)", 8650, 2134),
    (6, "Permits", "Distributor Permit", 69450, None),
    (6, "Permits", "Herbalist Permit", 870, 870),
    (6, "Permits", "Business Permit", 829316, 882125),
    (6, "Permits", "Other Permits", 8500, 4480),
]

NOTE_PAGES = {2: 21, 3: 22, 4: 23, 5: 24, 6: 24}

revenue_detail_df = pd.DataFrame([
    {"fiscal_year": YEAR, "note_number": note, "revenue_category": cat,
     "line_item": item, "amount_current_zmw": cur,
     "amount_prior_zmw": prev,
     "source_document": PDF_PATH.name, "source_page": NOTE_PAGES[note]}
    for note, cat, item, cur, prev in REVENUE_DETAIL
])

print(f"✅ Revenue detail: {len(revenue_detail_df)} rows")
revenue_detail_df.groupby("revenue_category").agg(
    items=("line_item", "count"),
    total=("amount_current_zmw", "sum"),
)

✅ Revenue detail: 53 rows


,items,total
revenue_category,,
Fees and Charges,25,18787346.0
Land Development Charges,2,372450.0
Levies,6,163188.0
Licences,7,220900.0
Local Taxes,4,325162.0
Permits,9,1522608.0


## 8. Validation

We validate extracted totals against the published figures on page 11.

In [11]:
# ============================================================
# VALIDATION
# ============================================================
print("=" * 70)
print(f"VALIDATION REPORT — Chirundu Town Council {YEAR}")
print("=" * 70)

checks = [
    ("Total Receipts (Statement)",
     receipts_df["amount_current_zmw"].sum(), 57868462),
    ("Total Payments (Statement)",
     payments_df["amount_current_zmw"].sum(), 61291747),
    ("LGEF Monthly Sum",
     lgef_df.loc[lgef_df["category"] == "LGEF Monthly Funding",
                 "amount_current_zmw"].sum(), 8833102),
    ("CDF Total Funding",
     cdf_df.loc[cdf_df["line_item"] == "Chirundu Constituency Funding",
                "amount_current_zmw"].sum(), 27261470),
]

for label, actual, expected in checks:
    diff = actual - expected
    status = "✅" if abs(diff) < 1 else "⚠️"
    print(f"{status} {label:<40} "
          f"actual={actual:>14,.2f}  expected={expected:>14,.2f}")

# --- Highlight the deficit ---
net = receipts_df["amount_current_zmw"].sum() - payments_df["amount_current_zmw"].sum()
print(f"\n📊 NET CASH MOVEMENT: K{net:,}")
if net < 0:
    print("⚠️  DEFICIT year — payments exceeded receipts")

print("\nAll checks complete.")

VALIDATION REPORT — Chirundu Town Council 2023
⚠️ Total Receipts (Statement)               actual= 57,868,463.00  expected= 57,868,462.00
✅ Total Payments (Statement)               actual= 61,291,747.00  expected= 61,291,747.00
✅ LGEF Monthly Sum                         actual=  8,833,101.76  expected=  8,833,102.00
✅ CDF Total Funding                        actual= 27,261,470.00  expected= 27,261,470.00

📊 NET CASH MOVEMENT: K-3,423,284
⚠️  DEFICIT year — payments exceeded receipts

All checks complete.


In [13]:
# ============================================================
# EXPORT — SINGLE UNIFIED CSV FOR 2023
# ============================================================
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("finances_output")

# --- Clean up obsolete per-table CSVs from earlier runs ---
keep_files = {
    f"chirundu_{YEAR}_raw_ocr.csv",
    f"db-unza26-csc4792-chirundu_{YEAR}_financials.csv",
}
for f in OUTPUT_DIR.iterdir():
    if f.is_file() and f.name not in keep_files and f.suffix == ".csv":
        f.unlink()

# --- Universal schema ---
UNIVERSAL_COLS = [
    "fiscal_year", "record_type", "section", "line_item",
    "amount_current_zmw", "amount_prior_zmw",
    "approved_budget_zmw", "actual_zmw",
    "variance_zmw", "variance_pct", "is_material_variance",
    "notes", "source_document", "source_page",
]


def normalize(df, record_type, section_col=None, notes_default=""):
    """Reshape a specific table into the unified schema."""
    out = pd.DataFrame()
    out["fiscal_year"] = df["fiscal_year"]
    out["record_type"] = record_type
    out["section"] = df[section_col] if section_col else ""
    out["line_item"] = df["line_item"]
    out["amount_current_zmw"] = df.get("amount_current_zmw")
    out["amount_prior_zmw"] = df.get("amount_prior_zmw")
    out["approved_budget_zmw"] = df.get("original_budget_zmw")
    out["actual_zmw"] = df.get("actual_zmw")
    out["variance_zmw"] = df.get("variance_zmw")
    out["variance_pct"] = df.get("variance_pct")
    out["is_material_variance"] = df.get("is_material_variance")
    out["notes"] = notes_default
    out["source_document"] = df["source_document"]
    out["source_page"] = df["source_page"]
    return out[UNIVERSAL_COLS]


# --- Normalize each of the 6 tables ---
receipts_norm = normalize(receipts_df, "receipt")
payments_norm = normalize(payments_df, "payment")
bva_norm = normalize(bva_df, "budget_vs_actual", section_col="type")
lgef_norm = normalize(lgef_df, "lgef", section_col="category")
cdf_norm = normalize(cdf_df, "cdf", section_col="category")
revenue_norm = normalize(revenue_detail_df, "revenue_detail",
                         section_col="revenue_category")

# --- Combine ---
combined_df = pd.concat(
    [receipts_norm, payments_norm, bva_norm,
     lgef_norm, cdf_norm, revenue_norm],
    ignore_index=True,
)

print(f"✅ Combined: {len(combined_df)} rows")
print(f"\nRows by record type:")
print(combined_df["record_type"].value_counts())

# --- Write one CSV ---
out_file = OUTPUT_DIR / (
    f"db-unza26-csc4792-chirundu_{YEAR}_financials.csv"
)
combined_df.to_csv(
    out_file,
    sep="|",
    index=False,
    encoding="utf-8-sig",
)

print(f"\n✅ Saved: {out_file.name} ({out_file.stat().st_size / 1024:.1f} KB)")

✅ Combined: 116 rows

Rows by record type:
record_type
revenue_detail      53
budget_vs_actual    15
lgef                15
cdf                 14
receipt             11
payment              8
Name: count, dtype: int64

✅ Saved: db-unza26-csc4792-chirundu_2023_financials.csv (15.8 KB)


In [14]:
# ============================================================
# VERIFY OUTPUT FOLDER
# ============================================================
from pathlib import Path

OUTPUT_DIR = Path("finances_output")

print(f"📁 {OUTPUT_DIR.resolve()}")
print("=" * 70)
files = sorted([f for f in OUTPUT_DIR.iterdir() if f.is_file()])
for f in files:
    print(f"  📄 {f.name:<60} {f.stat().st_size / 1024:>8.1f} KB")

print(f"\nTotal files: {len(files)}")

📁 C:\Users\USER\Videos\chitundu\finances_output
  📄 chirundu_2023_raw_ocr.csv                                        60.0 KB
  📄 db-unza26-csc4792-chirundu_2023_financials.csv                   15.8 KB

Total files: 2


In [15]:
# ============================================================
# EXPORT — SINGLE UNIFIED CSV (NO DELETION)
# ============================================================
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("finances_output")
OUTPUT_DIR.mkdir(exist_ok=True)

UNIVERSAL_COLS = [
    "fiscal_year", "record_type", "section", "line_item",
    "amount_current_zmw", "amount_prior_zmw",
    "approved_budget_zmw", "actual_zmw",
    "variance_zmw", "variance_pct", "is_material_variance",
    "notes", "source_document", "source_page",
]


def normalize(df, record_type, section_col=None, notes_default=""):
    out = pd.DataFrame()
    out["fiscal_year"] = df["fiscal_year"]
    out["record_type"] = record_type
    out["section"] = df[section_col] if section_col else ""
    out["line_item"] = df["line_item"]
    out["amount_current_zmw"] = df.get("amount_current_zmw")
    out["amount_prior_zmw"] = df.get("amount_prior_zmw")
    out["approved_budget_zmw"] = df.get("original_budget_zmw")
    out["actual_zmw"] = df.get("actual_zmw")
    out["variance_zmw"] = df.get("variance_zmw")
    out["variance_pct"] = df.get("variance_pct")
    out["is_material_variance"] = df.get("is_material_variance")
    out["notes"] = notes_default
    out["source_document"] = df["source_document"]
    out["source_page"] = df["source_page"]
    return out[UNIVERSAL_COLS]


# --- Normalize each table ---
receipts_norm = normalize(receipts_df, "receipt")
payments_norm = normalize(payments_df, "payment")
bva_norm = normalize(bva_df, "budget_vs_actual", section_col="type")
lgef_norm = normalize(lgef_df, "lgef", section_col="category")
cdf_norm = normalize(cdf_df, "cdf", section_col="category")
revenue_norm = normalize(revenue_detail_df, "revenue_detail",
                         section_col="revenue_category")

# --- Combine ---
combined_df = pd.concat(
    [receipts_norm, payments_norm, bva_norm,
     lgef_norm, cdf_norm, revenue_norm],
    ignore_index=True,
)

# --- Write this year's CSV (no deletion of other years) ---
out_file = OUTPUT_DIR / (
    f"db-unza26-csc4792-chirundu_{YEAR}_financials.csv"
)
combined_df.to_csv(
    out_file,
    sep="|",
    index=False,
    encoding="utf-8-sig",
)

print(f"✅ Saved: {out_file.name}")
print(f"   Rows: {len(combined_df)}")
print(f"   Size: {out_file.stat().st_size / 1024:.1f} KB")

✅ Saved: db-unza26-csc4792-chirundu_2023_financials.csv
   Rows: 116
   Size: 15.8 KB


In [17]:
# ============================================================
# CONVERT MISNAMED .xls (actually CSV) → PROPER .csv
# ============================================================
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("finances_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# --- Locate the .xls files ---
source_files = [
    Path.home() / "Downloads" / "db-unza26-csc4792-chirundu_2022_financials.xls",
    Path.home() / "Downloads" / "db-unza26-csc4792-chirundu_2023_financials.xls",
]

for src in source_files:
    if not src.exists():
        print(f"❌ Not found: {src}")
        continue

    # --- Extract year from filename ---
    year = "2022" if "2022" in src.name else "2023" if "2023" in src.name else "unknown"

    # --- Read as CSV with pipe separator ---
    # These are pipe-separated CSVs misnamed as .xls
    try:
        df = pd.read_csv(src, sep="|", encoding="utf-8-sig")
        # If the pipe count looks wrong, try other encodings
        if len(df.columns) < 5:
            df = pd.read_csv(src, sep="|", encoding="utf-8")
    except Exception as e:
        print(f"⚠️  First attempt failed for {src.name}: {e}")
        print("   Trying with latin-1 encoding...")
        df = pd.read_csv(src, sep="|", encoding="latin-1")

    # --- Clean column names (BOM artefact) ---
    df.columns = [
        c.replace("\ufeff", "").replace("ï»¿", "").strip()
        for c in df.columns
    ]

    # --- Write as proper pipe-separated CSV ---
    out = OUTPUT_DIR / f"db-unza26-csc4792-chirundu_{year}_financials.csv"
    df.to_csv(out, sep="|", index=False, encoding="utf-8-sig")

    print(f"✅ {src.name}")
    print(f"   → {out}")
    print(f"   {len(df)} rows, {len(df.columns)} columns")
    print(f"   Columns: {df.columns.tolist()}")
    print()

print("🎉 Conversion complete.")

✅ db-unza26-csc4792-chirundu_2022_financials.xls
   → finances_output\db-unza26-csc4792-chirundu_2022_financials.csv
   129 rows, 14 columns
   Columns: ['fiscal_year', 'record_type', 'section', 'line_item', 'amount_current_zmw', 'amount_prior_zmw', 'approved_budget_zmw', 'actual_zmw', 'variance_zmw', 'variance_pct', 'is_material_variance', 'notes', 'source_document', 'source_page']

✅ db-unza26-csc4792-chirundu_2023_financials.xls
   → finances_output\db-unza26-csc4792-chirundu_2023_financials.csv
   116 rows, 14 columns
   Columns: ['fiscal_year', 'record_type', 'section', 'line_item', 'amount_current_zmw', 'amount_prior_zmw', 'approved_budget_zmw', 'actual_zmw', 'variance_zmw', 'variance_pct', 'is_material_variance', 'notes', 'source_document', 'source_page']

🎉 Conversion complete.


In [18]:
# ============================================================
# VERIFY OUTPUT
# ============================================================
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("finances_output")

for year in [2022, 2023]:
    f = OUTPUT_DIR / f"db-unza26-csc4792-chirundu_{year}_financials.csv"

    if not f.exists():
        print(f"❌ Missing: {f.name}")
        continue

    print("=" * 70)
    print(f"📄 {f.name}")
    print("=" * 70)

    # Check the header has pipes
    with open(f, encoding="utf-8-sig") as fh:
        header = fh.readline().strip()

    pipe_count = header.count("|")
    print(f"Header: {header[:100]}...")
    print(f"Pipe count: {pipe_count}")

    # Load it
    df = pd.read_csv(f, sep="|")
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")
    print(f"Fiscal year(s): {df['fiscal_year'].unique().tolist()}")
    print(f"Record types: {df['record_type'].unique().tolist()}")
    print()

📄 db-unza26-csc4792-chirundu_2022_financials.csv
Header: fiscal_year|record_type|section|line_item|amount_current_zmw|amount_prior_zmw|approved_budget_zmw|ac...
Pipe count: 13
Rows: 129
Columns: 14
Fiscal year(s): [2022]
Record types: ['receipt', 'payment', 'budget_vs_actual', 'lgef', 'cdf', 'revenue_detail']

📄 db-unza26-csc4792-chirundu_2023_financials.csv
Header: fiscal_year|record_type|section|line_item|amount_current_zmw|amount_prior_zmw|approved_budget_zmw|ac...
Pipe count: 13
Rows: 116
Columns: 14
Fiscal year(s): [2023]
Record types: ['receipt', 'payment', 'budget_vs_actual', 'lgef', 'cdf', 'revenue_detail']



In [19]:
from pathlib import Path

for f in [
    Path.home() / "Downloads" / "db-unza26-csc4792-chirundu_2022_financials.xls",
    Path.home() / "Downloads" / "db-unza26-csc4792-chirundu_2023_financials.xls",
]:
    if f.exists():
        f.unlink()
        print(f"🗑️  Deleted: {f.name}")

🗑️  Deleted: db-unza26-csc4792-chirundu_2022_financials.xls
🗑️  Deleted: db-unza26-csc4792-chirundu_2023_financials.xls


In [20]:
# ============================================================
# CONVERT WHATSAPP-SHARED "XLS" (actually CSV or SpreadsheetML)
# ============================================================
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("finances_output")
OUTPUT_DIR.mkdir(exist_ok=True)

source_files = [
    Path.home() / "Downloads" / "db-unza26-csc4792-chirundu_2022_financials.xls",
    Path.home() / "Downloads" / "db-unza26-csc4792-chirundu_2023_financials.xls",
]


def read_robust(path):
    """Try every plausible read method."""
    # Method 1: Try reading as pipe-separated CSV
    try:
        df = pd.read_csv(path, sep="|", encoding="utf-8-sig")
        if len(df.columns) > 5:
            return df, "csv|utf-8-sig"
    except Exception:
        pass

    # Method 2: Try latin-1 encoding
    try:
        df = pd.read_csv(path, sep="|", encoding="latin-1")
        if len(df.columns) > 5:
            return df, "csv|latin-1"
    except Exception:
        pass

    # Method 3: Try comma separator
    try:
        df = pd.read_csv(path, sep=",", encoding="utf-8-sig")
        if len(df.columns) > 5:
            return df, "csv-comma|utf-8-sig"
    except Exception:
        pass

    # Method 4: Try Excel engine
    for engine in ("openpyxl", "xlrd"):
        try:
            df = pd.read_excel(path, engine=engine)
            if len(df.columns) > 5:
                return df, f"excel|{engine}"
        except Exception:
            pass

    # Method 5: Read raw bytes and inspect
    with open(path, "rb") as f:
        head = f.read(500)
    raise RuntimeError(
        f"All read methods failed. First bytes: {head[:200]!r}"
    )


for src in source_files:
    if not src.exists():
        print(f"❌ Not found: {src}")
        continue

    year = "2022" if "2022" in src.name else "2023" if "2023" in src.name else "unknown"

    try:
        df, method = read_robust(src)
    except Exception as e:
        print(f"❌ {src.name}: {e}")
        continue

    # --- Clean column names ---
    df.columns = [
        str(c).replace("\ufeff", "").replace("ï»¿", "").strip()
        for c in df.columns
    ]

    # --- Write as pipe-separated CSV ---
    out = OUTPUT_DIR / f"db-unza26-csc4792-chirundu_{year}_financials.csv"
    df.to_csv(out, sep="|", index=False, encoding="utf-8-sig")

    print(f"✅ {src.name}")
    print(f"   Read via: {method}")
    print(f"   → {out}")
    print(f"   {len(df)} rows, {len(df.columns)} cols")
    print(f"   Columns: {df.columns.tolist()}")
    print()

print("🎉 Done.")

❌ Not found: C:\Users\USER\Downloads\db-unza26-csc4792-chirundu_2022_financials.xls
✅ db-unza26-csc4792-chirundu_2023_financials.xls
   Read via: csv|utf-8-sig
   → finances_output\db-unza26-csc4792-chirundu_2023_financials.csv
   116 rows, 14 cols
   Columns: ['fiscal_year', 'record_type', 'section', 'line_item', 'amount_current_zmw', 'amount_prior_zmw', 'approved_budget_zmw', 'actual_zmw', 'variance_zmw', 'variance_pct', 'is_material_variance', 'notes', 'source_document', 'source_page']

🎉 Done.
